In [1]:
from pydantic import BaseModel, Field
from typing import Optional, List, Type, Dict, Any, Union, Literal, get_args
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import Runnable
from langgraph.types import Command
from src.haive.core.models.llm.base import LLMConfig, AzureLLMConfig


### **🔹 RouteChoice for Dynamic Routing**
class RouteChoice(BaseModel):
    """Defines a dynamic routing choice with a name and description."""
    name: str
    description: str


### **🔹 SupervisorRouterConfig (Manages Routing Choices & Dynamic Literals)**
class SupervisorRouterConfig(BaseModel):
    """Configuration for SupervisorRouter using structured output."""
    choices: List[RouteChoice] = Field(default=[], description="Available routing choices.")

    def create_literal_type(self) -> Type:
        """Dynamically creates a `Literal` type for the valid route choices."""
        choices = [choice.name for choice in self.choices] + ["END"]
        return Literal[*choices]

    def create_structured_output_model(self) -> Type[BaseModel]:
        """Create a structured output model for routing."""
        literal_type = self.create_literal_type()

        class RouteQuery(BaseModel):
            next: literal_type = Field(..., description="The next node to route to.")

        return RouteQuery

    def create_system_prompt(self) -> str:
        """Dynamically generates the system prompt based on choices."""
        choices_str = ", ".join(f'"{c.name}" ({c.description})' for c in self.choices)
        return f"""
        You are a router responsible for selecting the next step in a workflow.
        You MUST return one of the following options: {choices_str}.
        Your response MUST be a single JSON object with ONLY the "next" key and a valid value.
        Example output:
        {{
            "next": "vectorstore"
        }}
        If the task is completed you may respond with 'END'.
        """


### **🔹 `SupervisorRouter` Extends `AugLLMConfig` with Overridden `create_agent()`**
class SupervisorRouter(AugLLMConfig):
    """
    A dynamic supervisor that routes execution based on LLM decisions.
    It extends AugLLMConfig and **overrides `create_agent()`** to always use the dynamic router model.
    """

    router_config: SupervisorRouterConfig

    def __init__(self, **kwargs):
        super().__init__(**kwargs)  # ✅ Correctly inherit AugLLMConfig
        self.router_config = kwargs.get("router_config")

    @property
    def literal_type(self) -> Type:
        """Compute `Literal` type dynamically when accessed."""
        return self.router_config.create_literal_type()

    def create_agent(self) -> Runnable:
        """Overrides `create_agent()` to always use the dynamic routing model."""
        structured_output_model = self.router_config.create_structured_output_model()
        self.structured_output_model = structured_output_model  # Ensure consistency

        # **Force the LLM to use structured output with a dynamic system prompt**
        return (
            self.llm_config.instantiate_llm()
            .bind(system_message=self.router_config.create_system_prompt())
            .with_structured_output(structured_output_model)
        )

    def get_choices(self) -> List[str]:
        """Retrieve the list of available routing options."""
        return [choice.name for choice in self.router_config.choices]

    def format_input(self, input_state: Dict[str, Any]) -> List[HumanMessage]:
        """
        ✅ **Fix: Converts `input_state` into a valid input format for the LLM.**
        """

        # Extract user messages or default to a generic system message
        messages = input_state.get("messages", [])

        if not messages:
            return [SystemMessage(content="No messages provided.")]

        formatted_messages = []
        for message in messages:
            if isinstance(message, str):
                formatted_messages.append(HumanMessage(content=message))
            elif isinstance(message, dict) and "role" in message and "content" in message:
                if message["role"] == "system":
                    formatted_messages.append(SystemMessage(content=message["content"]))
                else:
                    formatted_messages.append(HumanMessage(content=message["content"]))
            else:
                raise ValueError(f"Invalid message format: {message}")

        return formatted_messages

    def route(self, input_state: Dict[str, Any]) -> Command:
        """
        Routes based on available choices using **`SupervisorRouter.create_agent().invoke()`**.
        ✅ **Fix: Calls `format_input()` to convert `input_state` before `.invoke()`**
        """

        formatted_input = self.format_input(input_state)

        # Invoke LLM agent for structured routing output
        response = self.create_agent().invoke(formatted_input)

        # Ensure response is a valid Enum choice
        next_node = response.next  # Keep as str

        # Dynamically set `Command[Literal[*members]]`
        return Command[Literal[*get_args(self.literal_type)]](update=input_state, goto=next_node)


### **🔹 Helper Function: Create a Supervisor Node for Graph Workflows**

def create_supervisor_node(supervisor) -> Callable[[State], Command]:
    """
    **Creates a dynamic supervisor node that selects the next action.**
    - ✅ Uses `ChatPromptTemplate` for structured system messages
    - ✅ Ensures routing logic is defined via `Command`
    """
    choices = supervisor.get_choices()
    valid_routes = choices + ["END"]  # Add "END" as valid

    chat_prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=supervisor.router_config.create_system_prompt()),
        HumanMessage(content="{messages}")  # Placeholder
    ])

    def supervisor_node(state: State) -> Command[Literal[*choices,"__end__"]]:
        """LLM-based router with implicit edges."""
        formatted_input = {"messages": state["messages"]}

        # ✅ Ensure input messages are correctly formatted as a list of BaseMessage
        if not isinstance(state["messages"], list) or not all(isinstance(m, BaseMessage) for m in state["messages"]):
            raise ValueError("State['messages'] must be a list of `BaseMessage` instances.")

        response = supervisor.create_agent().invoke(state["messages"])  # ✅ Fixed Input
        goto = response.next  # Extracts next node
        
        # ✅ Ensure `goto` is valid
        if goto not in valid_routes:
            print(f"⚠️ Invalid route '{goto}', defaulting to END.")
            goto=END
        elif goto=="END":
            goto = END

        return Command(update={"next": goto}, goto=goto)  # ✅ Implicit edge

    return supervisor_node


/home/will/Projects/haive/backend/haive/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:345: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'orm_mode' has been renamed to 'from_attributes'
  warnings.warn(message, UserWarning)


NameError: name 'AugLLMConfig' is not defined

In [ ]:
# Define routing config
router_config = SupervisorRouterConfig(
    choices=[
        RouteChoice(name="vectorstore", description="Retrieve from stored documents"),
        RouteChoice(name="web_search", description="Perform a live web search"),
    ]
)

# Create Supervisor Router
supervisor = SupervisorRouter(
    name="supervisor_router",
    llm_config=AzureLLMConfig(),
    router_config=router_config
)

print("Router Choices:", supervisor.get_choices())  
# Output: ['vectorstore', 'web_search']


In [ ]:
input_state = {"messages": [{"role": "user", "content": "Find me recent research on AI safety."}]}

# Run the router (calls `create_agent().invoke(input_state)`)
command = supervisor.route(input_state)
print("Route Decision:", command.goto)


In [2]:
print(supervisor)

NameError: name 'supervisor' is not defined

In [3]:
# Define routing config
router_config = SupervisorRouterConfig(
    choices=[
        RouteChoice(name="vectorstore", description="Retrieve from stored documents"),
        RouteChoice(name="web_search", description="Perform a live web search"),
    ]
)

# Create Supervisor Router
supervisor = SupervisorRouter(
    name="supervisor_router",
    llm_config=AzureLLMConfig(),
    router_config=router_config
)

print("Router Choices:", supervisor.get_choices())  
# Output: ['vectorstore', 'web_search']


NameError: name 'SupervisorRouter' is not defined

In [4]:
input_state = {
    "messages": [
        {"role": "user", "content": "Find me recent research on AI safety."}
    ]
}

# ✅ **Now `input_state` is correctly formatted before `invoke()`**
command = supervisor.route(input_state)
print("Route Decision:", command.goto)


NameError: name 'supervisor' is not defined

In [5]:
supervisor.router_config.choices.append(
    RouteChoice(name="database", description="Fetch structured data from SQL")
)
print(supervisor.get_choices())  # Output: ['vectorstore', 'web_search', 'database']


NameError: name 'supervisor' is not defined

In [6]:
input_state = {
    "messages": [
        {"role": "user", "content": "Fetch Data from DB"}
    ]
}

# ✅ **Now `input_state` is correctly formatted before `invoke()`**
command = supervisor.route(input_state)
print("Route Decision:", command.goto)


NameError: name 'supervisor' is not defined

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from typing import Dict, Any, Callable, TypedDict, List
from src.haive.core.models.llm.base import AzureLLMConfig


# ✅ Define State as a TypedDict
class State(TypedDict):
    messages: List[BaseMessage]
    next: str  # Stores the next action


# ✅ Supervisor Router Config (Dynamic Routing Choices)
router_config = SupervisorRouterConfig(
    choices=[
        RouteChoice(name="vectorstore", description="Retrieve from stored documents"),
        RouteChoice(name="web_search", description="Perform a live web search"),
    ]
)

# ✅ Create Supervisor Router
supervisor = SupervisorRouter(
    name="supervisor_router",
    llm_config=AzureLLMConfig(),
    router_config=router_config
)



# ✅ Define `vectorstore` & `web_search` example nodes
def vectorstore_node(state: State) -> State:
    print("🔎 Retrieving from vectorstore...")
    return {"messages": state["messages"] + [HumanMessage(content="Documents retrieved from vectorstore.")],}


def web_search_node(state: State) -> State:
    print("🌍 Performing live web search...")
    return {"messages": state["messages"] + [HumanMessage(content="Live search results retrieved.")]}


# ✅ Create LangGraph state machine
workflow = StateGraph(State)  # ✅ TypedDict State

# ✅ Add nodes dynamically
workflow.add_node("supervisor", create_supervisor_node(supervisor))
workflow.add_node("vectorstore", vectorstore_node)
workflow.add_node("web_search", web_search_node)
#workflow.add_node("END", lambda _: {"messages": []})  # ✅ Explicit END node

# ✅ **Define conditional edges from `supervisor`**
workflow.add_edge(START, "supervisor")
workflow.add_edge('vectorstore','supervisor')
workflow.add_edge('web_search','supervisor')
#workflow.add_conditional_edges(
#    "supervisor",
#    lambda state: state["next"],  # ✅ Uses state["next"] set by Command
#)

from langgraph.checkpoint.memory import MemorySaver
# ✅ Compile Graph (no explicit edges needed beyond supervisor)
graph = workflow.compile(checkpointer=MemorySaver())
config = {'configurable':{'thread_id':'1'}}

# ✅ Run test input
input_state = {"messages": [HumanMessage(content="Find me research on AI safety.")]}
updated_state = graph.invoke(input_state,config=config,debug=True)  # ✅ Returns full state, NOT Command

print("Route Decision:", updated_state["next"])  # ✅ Correctly extract next action


NameError: name 'SupervisorRouter' is not defined

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
print(workflow.nodes)

In [ ]:
print(graph.get_state(config))

In [ ]:
all_states = []
for state in graph.get_state_history(config):
    print(state)
    all_states.append(state)
    print("--")

In [ ]:
def database(state:State)->State:
    print('Performing Db Query')
    return {'messages':state['messages'] + [HumanMesage(content="Database Search results")]}
workflow.add_node('database',database)
#workflow.add_node('supervisor',create
#workflow.compile(checkpointer=MemorySaver)

In [ ]:
supervisor.router_config.choices.append(
    RouteChoice(name="database", description="Fetch structured data from SQL"))
print(dir(workflow))

In [ ]:
import inspect

def get_object_properties(
    obj, 
    attributes=None, 
    include_methods=False, 
    include_dunder=False, 
    pretty_print=False, 
    max_length=100
):
    """
    Extracts properties and methods of an object into a dictionary.

    Parameters:
    - obj: The object to inspect.
    - attributes (list, optional): Specific attributes to extract; defaults to all.
    - include_methods (bool): If True, includes callable methods; otherwise, only attributes.
    - include_dunder (bool): If True, includes dunder (__xx__) attributes; otherwise, excludes them.
    - pretty_print (bool): If True, prints a formatted output.
    - max_length (int): Truncates long values for readability in pretty-print mode.

    Returns:
    - dict: {attribute_name: (type, value)}
    """
    if attributes is None:
        attributes = dir(obj)  # Get all attributes if not specified

    properties = {}

    for attr in attributes:
        if not include_dunder and attr.startswith('__'):
            continue  # Skip dunder attributes unless explicitly included

        if hasattr(obj, attr):
            value = getattr(obj, attr)
            is_method = inspect.ismethod(value) or inspect.isfunction(value)
            
            if not include_methods and is_method:
                continue  # Skip methods if not requested
            
            value_str = repr(value)
            if len(value_str) > max_length:
                value_str = value_str[:max_length - 3] + '...'

            properties[attr] = (type(value).__name__, value_str)

    if pretty_print:
        header = f"{'Attribute':30} | {'Type':20} | Value"
        divider = "-" * len(header)
        print(header)
        print(divider)
        for attr, (typ, val) in properties.items():
            print(f"{attr:30} | {typ:20} | {val}")
    
    return properties


# Example Usage:
# workflow = typelanggraph.graph.StateGraph()
# get_object_properties(workflow, pretty_print=True)

get_object_properties(workflow,pretty_print=True)

In [ ]:
print(workflow.nodes)

In [ ]:
workflow.compile(checkpointer=MemorySaver())

In [ ]:
graph.invoke( {"messages": [HumanMessage(content="Query database to find the table")]},config=config)

In [8]:
all_states = []
for state in graph.get_state_history(config):
    print(state)
    all_states.append(state)
    print("--")

NameError: name 'graph' is not defined

In [9]:
history = graph.get_state_history(config)
print(history)

NameError: name 'graph' is not defined

In [10]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [11]:
from langgraph.types import Command
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from typing import Dict, Any, List, Type, get_args, Literal
from langgraph.graph import END


def create_supervisor_node(supervisor) -> Type[Callable[[Dict[str, Any]], Command]]:
    """
    Creates a LangGraph-compatible supervisor node that dynamically routes execution.

    - ✅ Uses `ChatPromptTemplate` for structured system messages
    - ✅ Ensures correct input formatting
    - ✅ Dynamically constructs `Literal` for `Command`
    """

    members = supervisor.get_choices()
    literal_type = supervisor.literal_type  # Dynamically computed `Literal[*members, "__end__"]`

    # ✅ **Correct ChatPromptTemplate System Message**
    chat_prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=supervisor.router_config.create_system_prompt()),
        HumanMessage(content="{messages}")  # Messages placeholder
    ])

    def supervisor_node(state: Dict[str, Any]) -> Command[literal_type]:
        """
        **An LLM-based router that selects the next action.**
        """
        formatted_input = {"messages": state["messages"]}

        # ✅ Invoke LLM for routing decision (calls `SupervisorRouter.create_agent().invoke()`)
        response = supervisor.create_agent().invoke(formatted_input)
        goto = response.next  # Extract the next node

        # ✅ If routing finishes, go to END
        if goto == "END":
            goto = END

        # ✅ Return structured Command with dynamic `Literal` type
        return Command[goto](update={"next": goto}, goto=goto)

    return supervisor_node


In [12]:
supervisor.router_config.choices.append(
    RouteChoice(name="database", description="Fetch structured data from SQL")
)
print(supervisor.get_choices())  # Output: ['vectorstore', 'web_search', 'database']


NameError: name 'supervisor' is not defined

In [13]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from typing import Dict, Any, Callable, TypedDict, List, Optional, Type, Union
from pydantic import BaseModel, Field
from src.haive.core.models.llm.base import AzureLLMConfig
from src.haive.core.engine.aug_llm import AugLLMConfig


# ✅ Define State as a TypedDict (Dynamically Modifiable)
class State(TypedDict):
    messages: List[BaseMessage]
    next: str


class SupervisorRouter(AugLLMConfig):
    """
    A dynamic supervisor router that extends AugLLMConfig.
    - ✅ Uses LLM-based decision-making
    - ✅ Handles structured output validation
    - ✅ Routes commands dynamically
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.router_config = kwargs.get("router_config")
        self._update_enum()

    def _update_enum(self):
        """Synchronize valid choices with the dynamic Enum."""
        DynamicRouteEnum.update_choices(self.get_choices())

    def get_choices(self) -> List[str]:
        """Retrieve available routing options."""
        return [choice.name for choice in self.router_config.choices]

    def create_agent(self):
        """Creates a structured LLM agent using AugLLMConfig."""
        return super().create_agent()

    def route(self, input_state: State) -> Command:
        """Routes based on structured LLM output."""
        response = self.create_agent().invoke(input_state["messages"])  # ✅ Fixed input
        next_node = response.next  # Extracts next node

        # ✅ Ensure valid routing
        if next_node not in self.get_choices() + ["END"]:
            print(f"⚠️ Invalid route '{next_node}', defaulting to END.")
            next_node = "END"

        return Command(update={"next": next_node}, goto=next_node)


In [14]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from typing import Dict, Any, Callable, TypedDict, List, Optional
from src.haive.core.models.llm.base import AzureLLMConfig


# ✅ Define State as a TypedDict
class State(TypedDict):
    messages: List[BaseMessage]
    next: str  # Stores the next action

class DynamicGraphManager:
    """Manages dynamic graph updates while preserving execution history."""

    def __init__(self, supervisor: SupervisorRouter):
        self.supervisor = supervisor
        self.state_history = []
        self.workflow = None
        self.graph = None  # ✅ Ensure `self.graph` is set
        self.create_workflow()

    def create_workflow(self):
        """Creates the base workflow with a supervisor node."""
        self.workflow = StateGraph(State)

        # ✅ Add core nodes
        self.workflow.add_node("supervisor", self.create_supervisor_node())
        self.workflow.add_node("vectorstore", self.vectorstore_node)
        self.workflow.add_node("web_search", self.web_search_node)
        self.workflow.add_node("END", lambda _: {"messages": []})  # ✅ Explicit END node

        # ✅ Define conditional routing (edges handled by Command)
        self.workflow.add_edge(START, "supervisor")

        # ✅ Compile the graph and store it
        self.graph = self.workflow.compile()

    def create_supervisor_node(self) -> Callable[[State], Command]:
        """Creates the dynamic supervisor node."""
        def supervisor_node(state: State) -> Command:
            command = self.supervisor.route(state)
            return command
        return supervisor_node

    def vectorstore_node(self, state: State) -> State:
        """Handles vectorstore retrieval."""
        return {"messages": state["messages"] + [HumanMessage(content="Vectorstore results retrieved.")], "next": "END"}

    def web_search_node(self, state: State) -> State:
        """Handles web search."""
        return {"messages": state["messages"] + [HumanMessage(content="Web search results retrieved.")], "next": "END"}

    def add_node(self, node_name: str, node_function: Callable[[State], State]):
        """Adds a new node dynamically and recompiles the graph."""
        if not self.workflow:
            raise ValueError("Graph not initialized. Run `create_workflow()` first.")

        print(f"➕ Adding node '{node_name}'")
        self.workflow.add_node(node_name, node_function)

        # ✅ Ensure new node routes back to supervisor if no `next` value is set
        self.workflow.add_conditional_edges(
            node_name,
            lambda state: state["next"] if "next" in state else self.supervisor.router_config.default_routing or "supervisor"
        )

        # ✅ Recompile the graph while preserving execution history
        self.graph = self.workflow.compile()  # ✅ FIX: Store new compiled graph

    def run(self, input_state: State):
        """Runs the graph while preserving history."""
        if not self.graph:
            raise ValueError("Graph is not initialized. Run `create_workflow()` first.")

        result = self.graph.invoke(input_state)
        self.state_history.append(result)  # ✅ Preserve history
        return result



In [15]:
# ✅ Supervisor Router Config
router_config = SupervisorRouterConfig(
    choices=[
        {"name": "vectorstore", "description": "Retrieve from stored documents"},
        {"name": "web_search", "description": "Perform a live web search"},
    ],
    default_routing="supervisor"  # ✅ Default to looping back
)

# ✅ Create Supervisor Router using AugLLMConfig
supervisor = SupervisorRouter(
    name="supervisor_router",
    llm_config=AzureLLMConfig(),
    router_config=router_config
)

# ✅ Initialize Graph Manager
graph_manager = DynamicGraphManager(supervisor)

# ✅ Run initial test input
input_state = {"messages": [HumanMessage(content="Find me research on AI safety.")]}
result = graph_manager.run(input_state)
print("📌 Route Decision:", result["next"])  # ✅ Should output "vectorstore" or "web_search"


Setting system prompt to 'You are a helpful assistant.'


ValueError: "SupervisorRouter" object has no field "router_config"

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from typing import Dict, Any, Callable, TypedDict, List, Optional
from src.haive.core.models.llm.base import AzureLLMConfig


# ✅ Define State as a TypedDict
class State(TypedDict):
    messages: List[BaseMessage]
    next: str  # Stores the next action


class DynamicGraphManager:
    """Manages dynamic graph updates while preserving execution history."""

    def __init__(self, supervisor: SupervisorRouter):
        self.supervisor = supervisor
        self.state_history = []
        self.workflow = None
        self.graph = None  # ✅ Ensure `self.graph` is set
        self.create_workflow()

    def create_workflow(self):
        """Creates the base workflow with a supervisor node."""
        self.workflow = StateGraph(State)

        # ✅ Add core nodes
        self.workflow.add_node("supervisor", self.create_supervisor_node())
        self.workflow.add_node("vectorstore", self.vectorstore_node)
        self.workflow.add_node("web_search", self.web_search_node)
        self.workflow.add_node("END", lambda _: {"messages": []})  # ✅ Explicit END node

        # ✅ Define conditional routing (edges handled by Command)
        self.workflow.add_edge(START, "supervisor")

        # ✅ Compile the graph and store it
        self.graph = self.workflow.compile()

    def create_supervisor_node(self) -> Callable[[State], Command]:
        """Creates the dynamic supervisor node."""
        def supervisor_node(state: State) -> Command:
            command = self.supervisor.route(state)
            return command
        return supervisor_node

    def vectorstore_node(self, state: State) -> State:
        """Handles vectorstore retrieval."""
        return {"messages": state["messages"] + [HumanMessage(content="Vectorstore results retrieved.")], "next": "END"}

    def web_search_node(self, state: State) -> State:
        """Handles web search."""
        return {"messages": state["messages"] + [HumanMessage(content="Web search results retrieved.")], "next": "END"}

    def add_node(self, node_name: str, node_function: Callable[[State], State]):
        """Adds a new node dynamically and recompiles the graph."""
        if not self.workflow:
            raise ValueError("Graph not initialized. Run `create_workflow()` first.")

        print(f"➕ Adding node '{node_name}'")
        self.workflow.add_node(node_name, node_function)

        # ✅ Ensure new node routes back to supervisor if no `next` value is set
        self.workflow.add_conditional_edges(
            node_name,
            lambda state: state["next"] if "next" in state else self.supervisor.router_config.default_routing or "supervisor"
        )

        # ✅ Recompile the graph while preserving execution history
        self.graph = self.workflow.compile()  # ✅ FIX: Store new compiled graph

    def run(self, input_state: State):
        """Runs the graph while preserving history."""
        if not self.graph:
            raise ValueError("Graph is not initialized. Run `create_workflow()` first.")

        result = self.graph.invoke(input_state)
        self.state_history.append(result)  # ✅ Preserve history
        return result


In [ ]:
# ✅ Supervisor Router Config
router_config = SupervisorRouterConfig(
    choices=[
        {"name": "vectorstore", "description": "Retrieve from stored documents"},
        {"name": "web_search", "description": "Perform a live web search"},
    ],
    default_routing="supervisor"  # ✅ Default to looping back
)

# ✅ Create Supervisor Router
supervisor = SupervisorRouter(
    name="supervisor_router",
    llm_config=AzureLLMConfig(),
    router_config=router_config
)

# ✅ Initialize Graph Manager
graph_manager = DynamicGraphManager(supervisor)

# ✅ Run initial test input
input_state = {"messages": [HumanMessage(content="Find me research on AI safety.")]}
result = graph_manager.run(input_state)
print("📌 Route Decision:", result["next"])  # ✅ Should output "vectorstore" or "web_search"
